# Text-To-Sign Production Colab

This notebook is the single supported Colab runner for the current project capabilities.

Dataset Build remains the current implemented public project stage. Sprint 3 Baseline Modeling is available here as a baseline-only operational workflow that consumes processed Dataset Build outputs.

Dataset Build reads fixed Google Drive raw inputs and publishes Dataset Build archives to `/content/drive/MyDrive/text-to-sign-production/artifacts/dataset-build/processed-v1/`.

Dataset Build keypoint inputs are the fixed Drive archives `train_2D_keypoints.tar.zst`, `val_2D_keypoints.tar.zst`, and `test_2D_keypoints.tar.zst`.

Sprint 3 Baseline Modeling publishes one Drive run root per `BASELINE_RUN_NAME` under `/content/drive/MyDrive/text-to-sign-production/artifacts/baseline-modeling/runs/<run_name>/` with `config/`, `checkpoints/`, `metrics/`, `qualitative/`, `record/`, and `archives/`.

Sprint 3 archive resume order is fixed for training, qualitative export, and record packaging: reuse already-extracted outputs, otherwise extract the deterministic archive, otherwise run the step and write both extracted outputs and the archive under the Drive run root.

The notebook is orchestration-only. Dataset and modeling logic stays in repository modules and this notebook calls stage workflows directly so progress bars can render live in Colab.


## 1. Environment And Repository Setup

This section installs the minimal Colab prerequisites, clones the repository, installs Python dependencies including the existing modeling extra, and prepares direct imports from `src/`.


In [ ]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Python: {sys.version.split()[0]}")

if shutil.which("zstd") is None:
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "zstd"], check=True)
else:
    print("zstd is already available.")

REPO_URL = "https://github.com/0xmillennium/text-to-sign-production.git"
REPO_REF = "master"
WORKTREE_ROOT = Path("/content/text-to-sign-production")
RUNTIME_ROOT = WORKTREE_ROOT.parent

RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
try:
    current_cwd = Path.cwd()
except FileNotFoundError:
    current_cwd = None

if current_cwd is None or current_cwd == WORKTREE_ROOT or WORKTREE_ROOT in current_cwd.parents:
    os.chdir(RUNTIME_ROOT)

if WORKTREE_ROOT.exists():
    shutil.rmtree(WORKTREE_ROOT)

subprocess.run(["git", "clone", REPO_URL, str(WORKTREE_ROOT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=WORKTREE_ROOT, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    cwd=WORKTREE_ROOT,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"],
    cwd=WORKTREE_ROOT,
    check=True,
)
os.chdir(WORKTREE_ROOT)

if str(WORKTREE_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(WORKTREE_ROOT / "src"))

current_revision = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=WORKTREE_ROOT,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
print(f"Repository ready at {WORKTREE_ROOT}")
print(f"Checked out revision: {current_revision}")
print("Python import path ready for direct library calls.")

## 2. Mount Google Drive And Select Run Settings

Edit `PIPELINE_SPLITS`, `BASELINE_RUN_NAME`, and `BASELINE_PANEL_SIZE` only. All input and output roots are fixed by repository workflows.


In [ ]:
from google.colab import drive

PIPELINE_SPLITS = ["train", "val", "test"]
BASELINE_RUN_NAME = "baseline-default"
BASELINE_PANEL_SIZE = 8

drive.mount("/content/drive", force_remount=False)
print(f"Dataset Build splits: {PIPELINE_SPLITS}")
print(f"Baseline run name: {BASELINE_RUN_NAME}")
print(f"Baseline panel size: {BASELINE_PANEL_SIZE}")
print("Dataset Build raw input root: /content/drive/MyDrive/text-to-sign-production/raw/how2sign")
print("Dataset Build local archive workspace: /content/how2sign_downloads")
print(
    "Dataset Build artifact root: "
    "/content/drive/MyDrive/text-to-sign-production/artifacts/dataset-build/processed-v1/"
)
print(
    "Baseline Modeling run root: "
    f"/content/drive/MyDrive/text-to-sign-production/artifacts/baseline-modeling/runs/{BASELINE_RUN_NAME}/"
)

## 3. Run Dataset Build

This calls the shared Dataset Build workflow directly. The workflow stages fixed Drive inputs, builds processed manifests and `.npz` samples, packages outputs, and publishes Dataset Build archives to the fixed Drive artifact path.


In [ ]:
from text_to_sign_production.data.constants import DEFAULT_FILTER_CONFIG_RELATIVE_PATH
from text_to_sign_production.workflows.dataset_build import run_dataset_build

dataset_result = run_dataset_build(
    splits=tuple(PIPELINE_SPLITS),
    filter_config_path=WORKTREE_ROOT / DEFAULT_FILTER_CONFIG_RELATIVE_PATH,
    input_mode="fixed_colab_drive",
    output_mode="fixed_colab_drive",
)

print(f"Dataset Build completed for splits: {', '.join(dataset_result.splits)}")
for item in dataset_result.staged_inputs:
    print(
        f"staged {item['split']}: translation={item['translation_path']} "
        f"keypoints={item['keypoints_path']}"
    )
for split in dataset_result.splits:
    raw_report = dataset_result.assumption_report["splits"][split]
    filter_report = dataset_result.filter_report["splits"][split]
    print(
        f"{split}: rows={raw_report['translation_row_count']} "
        f"matched={raw_report['matched_sample_count']} "
        f"kept={filter_report['kept_samples']} "
        f"dropped={filter_report['dropped_samples']}"
    )
for path in dataset_result.output_paths:
    print(f"published Dataset Build archive: {path}")

## 4. Locate Dataset Build Processed Outputs

Sprint 3 consumes processed Dataset Build manifests and `.npz` samples only. This step verifies the local processed outputs that the baseline config will read.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.modeling.training.config import load_baseline_training_config

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH

baseline_config = load_baseline_training_config(BASELINE_CONFIG_PATH)
print(f"Baseline config: {BASELINE_CONFIG_PATH}")
print(f"Train manifest: {baseline_config.data.train_manifest}")
print(f"Val manifest: {baseline_config.data.val_manifest}")
print(f"Default text backbone: {baseline_config.backbone.name}")

## 5. Prepare Sprint 3 Baseline Run Layout

This validates processed Dataset Build inputs and copies the baseline config into the Drive run root.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.workflows.baseline_modeling import (
    COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    run_baseline_modeling,
)

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH
BASELINE_RUN_NAME = globals().get("BASELINE_RUN_NAME", "baseline-default")
BASELINE_PANEL_SIZE = globals().get("BASELINE_PANEL_SIZE", 8)

prepare_result = run_baseline_modeling(
    step="prepare",
    config_path=BASELINE_CONFIG_PATH,
    run_name=BASELINE_RUN_NAME,
    artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    panel_size=BASELINE_PANEL_SIZE,
)

print(f"Prepared baseline run root: {prepare_result.layout.run_root}")
for step in prepare_result.steps:
    print(f"{step.step}: {step.action} -> {step.output_path}")

## 6. Resolve Or Run Sprint 3 Training Outputs

Training output resolution is Drive-aware: reuse `checkpoints/` and `metrics/` if they already exist, otherwise extract `archives/baseline_training_outputs.tar.zst`, otherwise run training and write the extracted outputs plus archive back to Drive.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.workflows.baseline_modeling import (
    COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    run_baseline_modeling,
)

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH
BASELINE_RUN_NAME = globals().get("BASELINE_RUN_NAME", "baseline-default")
BASELINE_PANEL_SIZE = globals().get("BASELINE_PANEL_SIZE", 8)

training_result = run_baseline_modeling(
    step="train",
    config_path=BASELINE_CONFIG_PATH,
    run_name=BASELINE_RUN_NAME,
    artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    panel_size=BASELINE_PANEL_SIZE,
)

for step in training_result.steps:
    print(f"{step.step}: {step.action} -> {step.output_path}")
    if step.archive_path is not None:
        print(f"archive: {step.archive_path}")

## 7. Resolve Or Run Sprint 3 Qualitative Export

Qualitative output resolution is Drive-aware: reuse `qualitative/` if it already exists, otherwise extract `archives/baseline_qualitative_outputs.tar.zst`, otherwise export the validation panel and write the extracted outputs plus archive back to Drive.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.workflows.baseline_modeling import (
    COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    run_baseline_modeling,
)

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH
BASELINE_RUN_NAME = globals().get("BASELINE_RUN_NAME", "baseline-default")
BASELINE_PANEL_SIZE = globals().get("BASELINE_PANEL_SIZE", 8)

qualitative_result = run_baseline_modeling(
    step="export-panel",
    config_path=BASELINE_CONFIG_PATH,
    run_name=BASELINE_RUN_NAME,
    artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    panel_size=BASELINE_PANEL_SIZE,
)

for step in qualitative_result.steps:
    print(f"{step.step}: {step.action} -> {step.output_path}")
    if step.archive_path is not None:
        print(f"archive: {step.archive_path}")

## 8. Resolve Or Build Sprint 3 Record Package

Record package resolution is Drive-aware: reuse `record/` if it already exists, otherwise extract `archives/baseline_record_package.tar.zst`, otherwise assemble the runtime-side package manifest and copied evidence references, then write the archive back to Drive.


In [ ]:
from text_to_sign_production.modeling.config import DEFAULT_BASELINE_CONFIG_PATH
from text_to_sign_production.workflows.baseline_modeling import (
    COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    run_baseline_modeling,
)

BASELINE_CONFIG_PATH = DEFAULT_BASELINE_CONFIG_PATH
BASELINE_RUN_NAME = globals().get("BASELINE_RUN_NAME", "baseline-default")
BASELINE_PANEL_SIZE = globals().get("BASELINE_PANEL_SIZE", 8)

package_result = run_baseline_modeling(
    step="package",
    config_path=BASELINE_CONFIG_PATH,
    run_name=BASELINE_RUN_NAME,
    artifact_runs_root=COLAB_BASELINE_ARTIFACT_RUNS_ROOT,
    panel_size=BASELINE_PANEL_SIZE,
)

for step in package_result.steps:
    print(f"{step.step}: {step.action} -> {step.output_path}")
    if step.archive_path is not None:
        print(f"archive: {step.archive_path}")

print(f"Baseline package manifest: {package_result.layout.package_manifest_path}")